In [1]:
#!/usr/bin/env python3

from __future__ import annotations

import argparse, csv, os, pathlib, re, sys, time
from dataclasses import dataclass
from typing import Iterator, List, Tuple

import pandas as pd
from bs4 import BeautifulSoup  # pip install beautifulsoup4
from selenium import webdriver  # pip install selenium~=4.21.0
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.remote.webelement import WebElement
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


###############################################################################
# 0. Config & helpers
###############################################################################

BLOCKWORDS = {
    "협찬", "체험단", "서포터즈", "공구", "원고료", "지원받아", "스폰", "서평", "AD", "ad",
    "광고", "홍보", "리뷰어", "샘플", "provided", "sponsored", "gifted",
}

# CSS 선택자 (Selectors)
BLOG_LINK_SELECTOR = "a.desc_inner"
PUBLISH_DATE_SEL   = "span.se_publishDate, span.date"
CONTENT_SEL        = "div.se-main-container, #postViewArea"

try:
    SCRIPT_DIR = pathlib.Path(__file__).resolve().parent
except NameError:
    SCRIPT_DIR = pathlib.Path.cwd()


###############################################################################
# 0‑1. Utility
###############################################################################

def sanitize(text: str) -> str:
    """공백을 하나로 줄이고 제어 문자를 제거하여 텍스트를 정리합니다."""
    clean = re.sub(r"[\x00-\x1f\u2028\u2029]+", " ", text)
    clean = re.sub(r"\s+", " ", clean).strip()
    return clean


def is_sponsored(text: str, allow: bool = False) -> bool:
    """텍스트에 협찬/광고 관련 단어가 포함되어 있는지 확인합니다."""
    if allow:
        return False
    lower = text.lower()
    return any(word.lower() in lower for word in BLOCKWORDS)


def locate_xlsx(path_like: str | pathlib.Path) -> pathlib.Path:
    """주어진 경로에서 엑셀 파일을 찾거나, 없으면 프로젝트 루트에서 검색합니다."""
    p = pathlib.Path(path_like)
    if p.expanduser().is_absolute():
        return p
    if p.exists():
        return p
    project_root = SCRIPT_DIR.parent
    hits = list(project_root.rglob(p.name))
    if hits:
        print(f"[정보] 엑셀 파일 위치 확인 → {hits[0].relative_to(project_root)}")
        return hits[0]
    raise FileNotFoundError(f"엑셀 파일 '{p}'를 찾을 수 없습니다 (검색 경로: {project_root})")

###############################################################################
# 1. Selenium boilerplate
###############################################################################

def make_driver(headless: bool = True) -> webdriver.Chrome:
    """Selenium Chrome 웹 드라이버 인스턴스를 생성합니다."""
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--lang=ko-KR")
    options.add_argument("--window-size=1280,1024")
    driver = webdriver.Chrome(options=options)
    driver.implicitly_wait(3)
    return driver

###############################################################################
# 2. Core crawler
###############################################################################

@dataclass
class Review:
    """수집된 리뷰 데이터를 저장하기 위한 데이터 클래스입니다."""
    search_keyword: str
    place_name: str
    address: str
    date: str
    text: str

    def as_tuple(self) -> Tuple[str, str, str, str, str]:
        """데이터를 CSV 저장을 위해 튜플 형태로 변환합니다."""
        return (
            self.search_keyword,
            self.place_name,
            self.address,
            self.date,
            self.text,
        )


def search_blog_links(driver: webdriver.Chrome, keyword: str, max_links: int) -> List[str]:
    """네이버 블로그의 2단계 페이지네이션을 처리하며 링크를 수집합니다."""
    links: list[str] = []
    
    print(f"  - 블로그 홈에서 '{keyword}' 검색 시작...")
    driver.get("https://section.blog.naver.com/")
    time.sleep(1.5)

    try:
        search_box = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "input.textbox[name='sectionBlogQuery']"))
        )
        search_box.send_keys(keyword)
        search_box.submit()
        time.sleep(1.5)
    except Exception as e:
        print(f"[경고] 블로그 섹션 페이지에서 검색을 수행하지 못했습니다: {e}", file=sys.stderr)
        return []

    while True:
        # 1. 현재 페이지에서 링크 수집
        elements = driver.find_elements(By.CSS_SELECTOR, BLOG_LINK_SELECTOR)
        new_links_found = False
        for a in elements:
            url = a.get_attribute("href")
            if url and url not in links:
                links.append(url)
                new_links_found = True
        
        # 목표 링크 수에 도달했거나 새 링크가 더 이상 없으면 종료
        if len(links) >= max_links:
            print(f"[정보] 목표 수량({max_links}개) 이상의 링크를 수집하여 중단합니다.")
            break
        if not new_links_found and len(links) > 0:
            print("[정보] 현재 페이지에서 더 이상 새로운 링크를 찾지 못해 중단합니다.")
            break

        # 2. 다음 페이지로 이동 (2단계 로직)
        try:
            # 2-1. 다음 페이지 번호(예: 2, 3, 4...)를 직접 찾아 클릭
            # 현재 페이지(strong 태그)를 찾고, 그 부모(span)의 바로 다음 형제(span) 안의 a 태그를 찾음
            pagination = driver.find_element(By.CLASS_NAME, "pagination")
            current_page_element = pagination.find_element(By.CSS_SELECTOR, "strong")
            next_page_link = current_page_element.find_element(By.XPATH, "./parent::span/following-sibling::span/a")
            
            print(f"    - {next_page_link.text} 페이지로 이동...")
            driver.execute_script("arguments[0].click();", next_page_link)
            time.sleep(1.2)
        except NoSuchElementException:
            # 2-2. 다음 페이지 번호가 없으면 '다음' 그룹 버튼을 찾아 클릭
            try:
                next_group_button = driver.find_element(By.CSS_SELECTOR, "a.button_next")
                print("    - '다음' 그룹 페이지로 이동...")
                driver.execute_script("arguments[0].click();", next_group_button)
                time.sleep(1.2)
            except NoSuchElementException:
                # 2-3. '다음' 그룹 버튼도 없으면 모든 페이지를 다 본 것이므로 종료
                print("[정보] 모든 페이지를 확인하여 수집을 중단합니다.")
                break
        except Exception as e:
            print(f"[경고] 페이지 이동 중 오류 발생: {e}", file=sys.stderr)
            break
            
    return links[:max_links]


def extract_post(driver: webdriver.Chrome, url: str) -> tuple[str | None, str | None]:
    """개별 블로그 포스트 URL에 접속하여 날짜와 본문 텍스트를 추출합니다."""
    try:
        driver.get(url)
        time.sleep(1.4)
        
        if "blog.naver.com" in driver.current_url and "PostView.naver" not in driver.current_url:
            try:
                driver.switch_to.frame("mainFrame")
            except Exception:
                pass
        
        soup = BeautifulSoup(driver.page_source, "html.parser")
        
        date_elt = soup.select_one(PUBLISH_DATE_SEL)
        date = sanitize(date_elt.get_text()) if date_elt else ""
        
        content_elt = soup.select_one(CONTENT_SEL)
        text = sanitize(content_elt.get_text(separator=" ")) if content_elt else ""
        
        driver.switch_to.default_content()
        
        return date, text
    except Exception as e:
        print(f"[경고] 포스트 추출 중 오류 발생: {e} – {url}", file=sys.stderr)
        return None, None


def crawl_reviews(keyword: str, place_name: str, address: str, *, max_posts: int = 5,
                  allow_sponsored: bool = False,
                  driver: webdriver.Chrome | None = None) -> Iterator[Review]:
    """주어진 키워드로 리뷰를 크롤링하는 전체 과정을 관리합니다."""
    close_driver = False
    if driver is None:
        driver = make_driver(headless=True)
        close_driver = True
    
    try:
        links = search_blog_links(driver, keyword, max_links=max_posts)
        
        print(f"  - 총 {len(links)}개의 블로그 링크 수집 완료.")
        for i, link in enumerate(links):
            print(f"    - 포스트 {i+1}/{len(links)} 추출 중...")
            date, text = extract_post(driver, link)
            if not (date and text):
                continue
            if is_sponsored(text, allow=allow_sponsored):
                print(f"      - 광고/협찬 포스트 제외: {link}")
                continue
            
            yield Review(keyword, place_name, address, date, text)
            
    finally:
        if close_driver:
            driver.quit()

###############################################################################
# 3. CLI batch pipeline
###############################################################################

def open_csv_writer(path: pathlib.Path, *, resume: bool) -> tuple[csv.writer, any]:
    """CSV 파일을 쓰기 또는 이어쓰기 모드로 엽니다."""
    mode = "a" if resume and path.exists() else "w"
    f = open(path, mode, newline="", encoding="utf-8-sig")
    writer = csv.writer(f)
    
    if mode == "w" or os.stat(path).st_size == 0:
        writer.writerow(["search_keyword", "place_name", "address", "date", "review_text"])
        f.flush()
        
    return writer, f


def load_done_set(path: pathlib.Path) -> set[str]:
    """이미 처리된 장소 목록을 CSV 파일에서 불러옵니다."""
    if not path.exists():
        return set()
    try:
        done_df = pd.read_csv(path, usecols=["place_name"])
        return set(done_df["place_name"].dropna().unique())
    except Exception:
        return set()


def run_batch(xlsx: pathlib.Path | str, output_csv: pathlib.Path, *, max_posts: int = 3,
              rate_limit: float = 1.0, allow_sponsored: bool = False,
              resume: bool = True):
    """엑셀 파일의 각 행을 순회하며 크롤링을 실행하는 메인 함수입니다."""
    xlsx_path = locate_xlsx(xlsx)
    df = pd.read_excel(xlsx_path)
    df["address"] = df["addr1"].fillna("") + " " + df["addr2"].fillna("")

    done_places = load_done_set(output_csv) if resume else set()
    if done_places:
        print(f"[정보] 이어쓰기 모드 활성화 — {output_csv} 파일에 이미 {len(done_places)}개의 장소가 처리됨")

    writer, fh = open_csv_writer(output_csv, resume=resume)
    driver = make_driver(headless=True)

    try:
        for i, row in df.iterrows():
            place_name = row["title"]
            if place_name in done_places:
                continue
            
            search_keyword = row["title"]
            address = sanitize(row["address"])
            
            print(f"[{i+1}/{len(df)}] ⏩ {search_keyword}")
            
            new_rows = 0
            for rev in crawl_reviews(
                search_keyword, place_name, address,
                max_posts=max_posts, allow_sponsored=allow_sponsored, driver=driver,
            ):
                writer.writerow(rev.as_tuple())
                fh.flush()
                new_rows += 1
            
            if new_rows == 0:
                writer.writerow([search_keyword, place_name, address, "", "(no review)"])
                fh.flush()
                
            time.sleep(rate_limit)
            
    finally:
        driver.quit()
        fh.close()
        print(f"✅ 작업 완료 — 데이터 저장 위치: {output_csv}")

###############################################################################
if __name__ == "__main__":
    # --- 여기서 파일 이름을 설정하세요 ---
    input_excel_file = '전국.xlsx'      # 👈 입력할 엑셀 파일 이름
    output_csv_file = 'naver_reviews.csv' # 👈 저장될 CSV 파일 이름

    # --- 아래 스크립트는 위에서 정의한 파일을 사용합니다 ---
    print(f"크롤링을 시작합니다...")
    print(f"입력 파일: {input_excel_file}")
    print(f"출력 파일: {output_csv_file}")
    
    run_batch(
        xlsx=input_excel_file,
        output_csv=pathlib.Path(output_csv_file),
        max_posts=100,
        rate_limit=1.0,
        allow_sponsored=False,
        resume=True,
    )

크롤링을 시작합니다...
입력 파일: 전국.xlsx
출력 파일: naver_reviews.csv
[정보] 이어쓰기 모드 활성화 — naver_reviews.csv 파일에 이미 194개의 장소가 처리됨
[200/51140] ⏩ 가야권역 소리마실 영농조합법인
  - 블로그 홈에서 '가야권역 소리마실 영농조합법인' 검색 시작...
    - '다음' 그룹 페이지로 이동...
    - '다음' 그룹 페이지로 이동...
    - '다음' 그룹 페이지로 이동...
    - '다음' 그룹 페이지로 이동...
    - '다음' 그룹 페이지로 이동...
    - '다음' 그룹 페이지로 이동...
    - '다음' 그룹 페이지로 이동...
    - '다음' 그룹 페이지로 이동...
    - '다음' 그룹 페이지로 이동...
[정보] 목표 수량(100개) 이상의 링크를 수집하여 중단합니다.
  - 총 100개의 블로그 링크 수집 완료.
    - 포스트 1/100 추출 중...
      - 광고/협찬 포스트 제외: https://blog.naver.com/yjh_reality/223924835205
    - 포스트 2/100 추출 중...
    - 포스트 3/100 추출 중...
    - 포스트 4/100 추출 중...
    - 포스트 5/100 추출 중...
      - 광고/협찬 포스트 제외: https://blog.naver.com/yooniji_/223925484055
    - 포스트 6/100 추출 중...
      - 광고/협찬 포스트 제외: https://blog.naver.com/lass3679/223925360614
    - 포스트 7/100 추출 중...
      - 광고/협찬 포스트 제외: https://blog.naver.com/orange_smurf/223925341464
    - 포스트 8/100 추출 중...
    - 포스트 9/100 추출 중...
      - 광고/협찬 포스트 제외: https://blog.nav

KeyboardInterrupt: 

In [1]:
#!/usr/bin/env python3

from __future__ import annotations

import argparse, csv, os, pathlib, re, sys, time
from dataclasses import dataclass
from typing import Iterator, List, Tuple

import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

###############################################################################
# 0. Config & helpers
###############################################################################

BLOCKWORDS = {
    "협찬", "체험단", "서포터즈", "공구", "원고료", "지원받아", "스폰", "서평", "AD", "ad",
    "광고", "홍보", "리뷰어", "샘플", "provided", "sponsored", "gifted",
}

# CSS Selectors for Naver Map and Blog pages
MAP_SEARCH_IFRAME   = "iframe#searchIframe"
MAP_RESULTS_LIST    = "div#_pcmap_list_scroll_container"
MAP_RESULT_ITEM     = "li._3t81n"
MAP_ITEM_NAME       = "span.YwYLL"
MAP_ITEM_ADDR       = "span.kRl4e"
PLACE_REVIEWS_TAB   = 'a[href*="/review"]' # Find an <a> tag whose href contains "/review"
PLACE_BLOG_TAB      = 'a[role="tab"][aria-selected="false"]' # Tab for '블로그'
BLOG_LINK_SELECTOR  = "a.t9x3A" # Link to the actual blog post in the review list
PUBLISH_DATE_SEL    = "span.se_publishDate, span.date"
CONTENT_SEL         = "div.se-main-container, div#postViewArea"

try:
    SCRIPT_DIR = pathlib.Path(__file__).resolve().parent
except NameError:
    SCRIPT_DIR = pathlib.Path.cwd()

###############################################################################
# 0-1. Utility
###############################################################################

def sanitize(text: str) -> str:
    """Collapse whitespace & strip control chars."""
    clean = re.sub(r"[\x00-\x1f\u2028\u2029]+", " ", text)
    clean = re.sub(r"\s+", " ", clean).strip()
    return clean

def is_sponsored(text: str, allow: bool = False) -> bool:
    """Return True iff text seems paid/sponsored."""
    if allow:
        return False
    lower = text.lower()
    return any(word.lower() in lower for word in BLOCKWORDS)

def locate_xlsx(path_like: str | pathlib.Path) -> pathlib.Path:
    """Resolve *path_like*; if not found, search under project root."""
    p = pathlib.Path(path_like)
    if p.expanduser().is_absolute():
        return p
    if p.exists():
        return p
    project_root = SCRIPT_DIR.parent
    hits = list(project_root.rglob(p.name))
    if hits:
        print(f"[info] Located Excel → {hits[0].relative_to(project_root)}")
        return hits[0]
    raise FileNotFoundError(f"Excel file '{p}' not found (searched under {project_root})")

###############################################################################
# 1. Selenium boilerplate
###############################################################################

def make_driver(headless: bool = True) -> webdriver.Chrome:
    """Creates a Selenium WebDriver instance."""
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--lang=ko-KR")
    options.add_argument("--window-size=1280,1024")
    driver = webdriver.Chrome(options=options)
    driver.implicitly_wait(3)
    return driver

###############################################################################
# 2. Core crawler
###############################################################################

@dataclass
class Review:
    search_keyword: str
    place_name: str
    address: str
    date: str
    text: str

    def as_tuple(self) -> Tuple[str, str, str, str, str]:
        return (
            self.search_keyword,
            self.place_name,
            self.address,
            self.date,
            self.text,
        )

def search_links_via_map(driver: webdriver.Chrome, place_name: str, address: str, max_posts: int) -> List[str]:
    """
    Finds blog review links via Naver Map for better accuracy.
    Handles both direct-hit pages and search list pages.
    """
    print(f"  🗺️  Searching Naver Map for '{place_name}'...")
    links = []
    search_url = f"https://map.naver.com/p/search/{place_name}"
    driver.get(search_url)
    wait = WebDriverWait(driver, 8) # Use a slightly shorter wait time for the initial check

    try:
        # --- NEW LOGIC: Tries to handle BOTH page types ---
        # Path A: Check for a direct hit page first (no iframe)
        print("  ➡️  Checking for a direct-hit page...")
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, PLACE_REVIEWS_TAB)))
        print("  ✔️  Direct-hit page confirmed. Proceeding to reviews.")

    except TimeoutException:
        # Path B: If direct hit fails, assume it's a list page and switch to iframe
        print("  ➡️  Direct-hit failed. Assuming search list page, switching to iframe...")
        try:
            wait_long = WebDriverWait(driver, 10)
            wait_long.until(EC.frame_to_be_available_and_switch_to_it((By.CSS_SELECTOR, MAP_SEARCH_IFRAME)))
            
            # Find the correct place by matching address
            wait_long.until(EC.presence_of_element_located((By.CSS_SELECTOR, MAP_RESULTS_LIST)))
            results = driver.find_elements(By.CSS_SELECTOR, MAP_RESULT_ITEM)
            
            found_place = None
            addr_short = " ".join(address.split()[:3]) # Compare first 3 words of address
            for result in results:
                try:
                    map_name = result.find_element(By.CSS_SELECTOR, MAP_ITEM_NAME).text
                    map_addr = result.find_element(By.CSS_SELECTOR, MAP_ITEM_ADDR).text
                    if place_name in map_name and addr_short in map_addr:
                        print(f"  ✔️  Found matching place in list: {map_name} ({map_addr})")
                        found_place = result
                        break
                except NoSuchElementException:
                    continue

            if not found_place:
                print(f"  ⚠️  Could not find a matching place for '{place_name}' in the list.")
                driver.switch_to.default_content()
                return []
            
            # Click the place to load its details
            found_place.click()
            time.sleep(1.5) # Wait for details to load

        except Exception as e:
            print(f"  ⚠️  An error occurred during map search for '{place_name}': {e}", file=sys.stderr)
            driver.switch_to.default_content()
            return []

    # --- COMMON LOGIC: From here, the process is the same for both paths ---
    try:
        print("  📝  Navigating to blog reviews tab...")
        # Click "리뷰" tab
        # Use WebDriverWait for robustness
        review_tab_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, PLACE_REVIEWS_TAB))
        )
        review_tab_button.click()
        time.sleep(1.5)

        # Click "블로그리뷰" tab
        blog_tab_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "//span[contains(text(), '블로그리뷰')]/ancestor::a"))
        )
        blog_tab_button.click()
        time.sleep(1.5)

        # Scrape blog post links
        review_elements = driver.find_elements(By.CSS_SELECTOR, BLOG_LINK_SELECTOR)
        for el in review_elements[:max_posts]:
            link = el.get_attribute('href')
            if link and link not in links:
                links.append(link)
        
        driver.switch_to.default_content() # Switch back out of iframe context if we were in one
        return links
        
    except Exception as e:
        print(f"  ⚠️  Failed to navigate to reviews and extract links: {e}", file=sys.stderr)
        driver.switch_to.default_content()
        return []

def extract_post(driver: webdriver.Chrome, url: str) -> tuple[str | None, str | None]:
    """Extracts date and text from a single blog post URL."""
    try:
        driver.get(url)
        time.sleep(1.2)
        # Handle blog.naver.com's mainFrame if it exists
        if "blog.naver.com" in driver.current_url:
            try:
                driver.switch_to.frame("mainFrame")
            except Exception:
                pass # No frame to switch to, continue
        
        soup = BeautifulSoup(driver.page_source, "html.parser")
        date_elt = soup.select_one(PUBLISH_DATE_SEL)
        date = sanitize(date_elt.get_text()) if date_elt else ""
        content_elt = soup.select_one(CONTENT_SEL)
        text = sanitize(content_elt.get_text(separator=" ")) if content_elt else ""
        return date, text
    except Exception as e:
        print(f"[warn] Failed to extract {url}: {e}", file=sys.stderr)
        return None, None

def crawl_reviews(
    keyword: str, place_name: str, address: str, *,
    max_posts: int = 5,
    allow_sponsored: bool = False,
    driver: webdriver.Chrome | None = None
) -> Iterator[Review]:
    """Crawls reviews for a single place."""
    close_driver = False
    if driver is None:
        driver = make_driver(headless=True)
        close_driver = True
    
    try:
        # Use the new map-based search function
        links = search_links_via_map(driver, keyword, address, max_posts=max_posts)
        
        if not links:
            print(f"  텅! No blog links found for '{keyword}' via map search.")
            return

        print(f"  Found {len(links)} blog links. Extracting content...")
        for link in links:
            date, text = extract_post(driver, link)
            if not (date and text):
                continue
            if is_sponsored(text, allow=allow_sponsored):
                print(f"  🚫 Skipping sponsored post: {link}")
                continue
            yield Review(keyword, place_name, address, date, text)
    finally:
        if close_driver:
            driver.quit()

###############################################################################
# 3. CLI batch pipeline
###############################################################################

def open_csv_writer(path: pathlib.Path, *, resume: bool) -> tuple[csv.writer, any]:
    """Opens a CSV file in append mode."""
    mode = "a" if resume and path.exists() else "w"
    f = open(path, mode, newline="", encoding="utf-8-sig")
    writer = csv.writer(f)
    if mode == "w" or os.stat(path).st_size == 0:
        writer.writerow(["search_keyword", "place_name", "address", "date", "review_text"])
        f.flush()
    return writer, f

def load_done_set(path: pathlib.Path) -> set[str]:
    """Loads already processed places from the CSV to allow resuming."""
    if not path.exists():
        return set()
    try:
        done_df = pd.read_csv(path, usecols=["place_name"])
        return set(done_df["place_name"].dropna().unique())
    except Exception:
        return set()

def run_batch(
    xlsx: pathlib.Path | str, output_csv: pathlib.Path, *,
    max_posts: int = 3,
    rate_limit: float = 1.0,
    allow_sponsored: bool = False,
    resume: bool = True
):
    """Main function to run the batch crawling process."""
    xlsx_path = locate_xlsx(xlsx)
    df = pd.read_excel(xlsx_path)
    # Ensure address columns exist, fill missing with empty string
    if 'addr1' not in df.columns: df['addr1'] = ""
    if 'addr2' not in df.columns: df['addr2'] = ""
    df["address"] = df["addr1"].fillna("") + " " + df["addr2"].fillna("")

    done_places = load_done_set(output_csv) if resume else set()
    if done_places:
        print(f"[info] Resuming — {len(done_places)} places already in {output_csv}")

    writer, fh = open_csv_writer(output_csv, resume=resume)
    driver = make_driver(headless=True) # Use one driver for the whole session
    
    try:
        for i, row in df.iterrows():
            place_title = row["title"]
            if place_title in done_places:
                continue
            
            addr = sanitize(row["address"])
            print(f"\n[{i+1}/{len(df)}] ⏩ Processing: {place_title}")
            
            new_rows = 0
            for rev in crawl_reviews(
                place_title, place_title, addr,
                max_posts=max_posts,
                allow_sponsored=allow_sponsored,
                driver=driver,
            ):
                writer.writerow(rev.as_tuple())
                fh.flush() # Immediately write to disk
                new_rows += 1
            
            if new_rows == 0:
                # Write a placeholder row to mark this place as 'done'
                writer.writerow([place_title, place_title, addr, "", "(no review found via map)"])
                fh.flush()
                
            time.sleep(rate_limit) # Be respectful to the server
    finally:
        driver.quit()
        fh.close()
        print(f"\n✅ 작업 완료 — 데이터 저장 위치: {output_csv}")

###############################################################################
# Main execution block
###############################################################################

if __name__ == "__main__":
    # --- Define your files and settings here ---
    input_excel_file = '전국.xlsx'  # 👈 CHANGE THIS to your Excel file name
    output_csv_file = 'naver_reviews_map.csv' # 👈 CHANGE THIS to your desired output file name
    
    # --- The script will use the files and settings you defined above ---
    print(f"Starting batch process...")
    print(f"Input file: {input_excel_file}")
    print(f"Output file: {output_csv_file}")
    
    run_batch(
        xlsx=input_excel_file,
        output_csv=pathlib.Path(output_csv_file),
        max_posts=5,           # 각 장소당 수집할 최대 블로그 리뷰 수
        rate_limit=1.5,        # 각 장소 처리 후 대기 시간 (초)
        allow_sponsored=False, # "협찬" 등 광고성 포스트 포함 여부
        resume=True,           # True로 두면 이어서 작업
    )

Starting batch process...
Input file: 전국.xlsx
Output file: naver_reviews_map.csv
[info] Resuming — 1 places already in naver_reviews_map.csv

[2/51140] ⏩ Processing: 가가와
  🗺️  Searching Naver Map for '가가와'...
  ➡️  Checking for a direct-hit page...
  ➡️  Direct-hit failed. Assuming search list page, switching to iframe...


  ⚠️  An error occurred during map search for '가가와': Message: 
Stacktrace:
	GetHandleVerifier [0x0x7ff751446f95+76917]
	GetHandleVerifier [0x0x7ff751446ff0+77008]
	(No symbol) [0x0x7ff7511f9dea]
	(No symbol) [0x0x7ff751250256]
	(No symbol) [0x0x7ff75125050c]
	(No symbol) [0x0x7ff7512a3887]
	(No symbol) [0x0x7ff7512784af]
	(No symbol) [0x0x7ff7512a065c]
	(No symbol) [0x0x7ff751278243]
	(No symbol) [0x0x7ff751241431]
	(No symbol) [0x0x7ff7512421c3]
	GetHandleVerifier [0x0x7ff75171d2cd+3051437]
	GetHandleVerifier [0x0x7ff751717923+3028483]
	GetHandleVerifier [0x0x7ff7517358bd+3151261]
	GetHandleVerifier [0x0x7ff75146185e+185662]
	GetHandleVerifier [0x0x7ff75146971f+218111]
	GetHandleVerifier [0x0x7ff75144fb14+112628]
	GetHandleVerifier [0x0x7ff75144fcc9+113065]
	GetHandleVerifier [0x0x7ff751436c98+10616]
	BaseThreadInitThunk [0x0x7ffdce00e8d7+23]
	RtlUserThreadStart [0x0x7ffdcfbbc34c+44]



  텅! No blog links found for '가가와' via map search.

[3/51140] ⏩ Processing: 가가책방
  🗺️  Searching Naver Map for '가가책방'...
  ➡️  Checking for a direct-hit page...
  ➡️  Direct-hit failed. Assuming search list page, switching to iframe...


  ⚠️  An error occurred during map search for '가가책방': Message: 
Stacktrace:
	GetHandleVerifier [0x0x7ff751446f95+76917]
	GetHandleVerifier [0x0x7ff751446ff0+77008]
	(No symbol) [0x0x7ff7511f9dea]
	(No symbol) [0x0x7ff751250256]
	(No symbol) [0x0x7ff75125050c]
	(No symbol) [0x0x7ff7512a3887]
	(No symbol) [0x0x7ff7512784af]
	(No symbol) [0x0x7ff7512a065c]
	(No symbol) [0x0x7ff751278243]
	(No symbol) [0x0x7ff751241431]
	(No symbol) [0x0x7ff7512421c3]
	GetHandleVerifier [0x0x7ff75171d2cd+3051437]
	GetHandleVerifier [0x0x7ff751717923+3028483]
	GetHandleVerifier [0x0x7ff7517358bd+3151261]
	GetHandleVerifier [0x0x7ff75146185e+185662]
	GetHandleVerifier [0x0x7ff75146971f+218111]
	GetHandleVerifier [0x0x7ff75144fb14+112628]
	GetHandleVerifier [0x0x7ff75144fcc9+113065]
	GetHandleVerifier [0x0x7ff751436c98+10616]
	BaseThreadInitThunk [0x0x7ffdce00e8d7+23]
	RtlUserThreadStart [0x0x7ffdcfbbc34c+44]



  텅! No blog links found for '가가책방' via map search.

[4/51140] ⏩ Processing: 가거도(소흑산도)
  🗺️  Searching Naver Map for '가거도(소흑산도)'...
  ➡️  Checking for a direct-hit page...
  ➡️  Direct-hit failed. Assuming search list page, switching to iframe...


...

  ⚠️  An error occurred during map search for '가고파생삼겹구이': Message: 
Stacktrace:
	GetHandleVerifier [0x0x7ff751446f95+76917]
	GetHandleVerifier [0x0x7ff751446ff0+77008]
	(No symbol) [0x0x7ff7511f9dea]
	(No symbol) [0x0x7ff751250256]
	(No symbol) [0x0x7ff75125050c]
	(No symbol) [0x0x7ff7512a3887]
	(No symbol) [0x0x7ff7512784af]
	(No symbol) [0x0x7ff7512a065c]
	(No symbol) [0x0x7ff751278243]
	(No symbol) [0x0x7ff751241431]
	(No symbol) [0x0x7ff7512421c3]
	GetHandleVerifier [0x0x7ff75171d2cd+3051437]
	GetHandleVerifier [0x0x7ff751717923+3028483]
	GetHandleVerifier [0x0x7ff7517358bd+3151261]
	GetHandleVerifier [0x0x7ff75146185e+185662]
	GetHandleVerifier [0x0x7ff75146971f+218111]
	GetHandleVerifier [0x0x7ff75144fb14+112628]
	GetHandleVerifier [0x0x7ff75144fcc9+113065]
	GetHandleVerifier [0x0x7ff751436c98+10616]
	BaseThreadInitThunk [0x0x7ffdce00e8d7+23]
	RtlUserThreadStart [0x0x7ffdcfbbc34c+44]



  텅! No blog links found for '가고파생삼겹구이' via map search.

[13/51140] ⏩ Processing: 가고파식당
  🗺️  Searching Naver Map for '가고파식당'...
  ➡️  Checking for a direct-hit page...
  ➡️  Direct-hit failed. Assuming search list page, switching to iframe...


  ⚠️  An error occurred during map search for '가고파식당': Message: 
Stacktrace:
	GetHandleVerifier [0x0x7ff751446f95+76917]
	GetHandleVerifier [0x0x7ff751446ff0+77008]
	(No symbol) [0x0x7ff7511f9dea]
	(No symbol) [0x0x7ff751250256]
	(No symbol) [0x0x7ff75125050c]
	(No symbol) [0x0x7ff7512a3887]
	(No symbol) [0x0x7ff7512784af]
	(No symbol) [0x0x7ff7512a065c]
	(No symbol) [0x0x7ff751278243]
	(No symbol) [0x0x7ff751241431]
	(No symbol) [0x0x7ff7512421c3]
	GetHandleVerifier [0x0x7ff75171d2cd+3051437]
	GetHandleVerifier [0x0x7ff751717923+3028483]
	GetHandleVerifier [0x0x7ff7517358bd+3151261]
	GetHandleVerifier [0x0x7ff75146185e+185662]
	GetHandleVerifier [0x0x7ff75146971f+218111]
	GetHandleVerifier [0x0x7ff75144fb14+112628]
	GetHandleVerifier [0x0x7ff75144fcc9+113065]
	GetHandleVerifier [0x0x7ff751436c98+10616]
	BaseThreadInitThunk [0x0x7ffdce00e8d7+23]
	RtlUserThreadStart [0x0x7ffdcfbbc34c+44]



  텅! No blog links found for '가고파식당' via map search.

[14/51140] ⏩ Processing: 가고파식당
  🗺️  Searching Naver Map for '가고파식당'...
  ➡️  Checking for a direct-hit page...

✅ 작업 완료 — 데이터 저장 위치: naver_reviews_map.csv


KeyboardInterrupt: 